In [ ]:
from cropspectai import config
from cropspectai.data_analysis import preprocessing
from cropspectai.modeling.evaluate import evaluate_model

import pandas as pd

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


# --------------------------------------------------
# Load data
# --------------------------------------------------

spectral_length = "as7343" # "full" or "1000" or "as7343"
target = "cab" # "cab" or None
indices = "all" # "all" or "established" or None

if spectral_length == "as7343":
    if target == "cab":
        if indices == "all": # all engineered features
            test_path = (
                        config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "test"
                        / "test_all_features.parquet"
                    )
#         elif indices == "established": # only established new features
#             train_path = (
#                         config.PROCESSED_PARQUET_DATA_SPLITS_PATH
#                         / target
#                         / "train_established_features.parquet"
#                     ) 
#         elif indices == None: # no engineered features
#             train_path = (
#                         config.PROCESSED_PARQUET_DATA_SPLITS_PATH
#                         / target
#                         / "train.parquet"
#                     )       
# elif spectral_length == "full" or "1000":
#     train_path = (
#         config.RAW_PARQUET_DATA_SPLITS_PATH
#         / "train.parquet"
#     )

df = pd.read_parquet(test_path)
print("Shape of imported data: ", df.shape)

Shape of imported data:  (396, 41)


/Users/sharanashivanand/git_repos/dsr/cropspectai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --------------------------------------------------
# Get feature and target column names
# --------------------------------------------------

X, y = preprocessing.split_dataset_into_X_y(df, target)

targets = y.columns.tolist()

print("Targets:")
print(targets)

print("Features:")
print(X.columns.tolist())

Targets:
['cab']
Features:
['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343', 'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343', 'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343', 'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343', 'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343', 'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd', 'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio', 'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio']


In [3]:
top_indices = ['mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343', 'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343'] 
# top_indices = ['mndvi_as7343', 'sipi_as7343']
# top_indices = ['mndvi_as7343']
# top_indices = ['red_edge_green_nd']
# top_indices = ['red_green_nd'] # 1
# top_indices = ['psri_as7343'] # 2
# top_indices = ['datt1_as7343']   
AS7343_CHANNELS = list(config.AS7343_CHANNELS.keys())
full_list = AS7343_CHANNELS + top_indices
X = X[full_list].copy()
features = X.columns.tolist()

print("Features:")
print(features)
print(X.shape)

df = pd.concat([X, y], axis = 1).copy()
print(df.shape)

Features:
['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343', 'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343']
(396, 19)
(396, 20)


In [4]:
df_target = preprocessing.drop_duplicates(
        df,
        target
    ).copy()

(Before) Shape of dataframe: (396, 20)
Dropping duplicate rows
Duplicate rows: 0
(After) Shape of dataframe: (396, 20)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (396, 20)


## load best xgb model

In [5]:
file_name = f"xgb_v_01.joblib"
model_path = config.MODELS_DIR / file_name

In [6]:
evaluate_model(X, y, model_path)

{'MAE (test)': 6.938453197479248,
 'MAPE (test) [%]': 22.039984166622162,
 'RMSE (test)': 9.477968215942383,
 'R² (test)': 0.5113708972930908}